# RR ETDs to MARC

This script retrieves OAI metadata for the ETDs located in WVU's Research Repository and uses an XSLT script to transform the metadata into MARC/XML. It can then perform an API call to validate each record and upload the valid records into WMS.

Inputs:
- Date range for retrieval of records from the respository
- Subfolder name for saving output
- y/n on whether to upload to OCLC via API

Outputs:
- XML:
  - OAI metadata that is input to the XSLT script
  - Complete MARC output file
  - Invalid MARC records
  - Valid MARC records

- XLSX:
  - Embargoed records that were not processed or uploaded
  - List of invalid records that were not uploaded
  - List of records missing DOIs that were not uploaded
  - List of records for campus access only ETDs
  - List of records successfully uploaded


## Dependencies

In [ ]:
# connect to Google Drive. The XSLT stylesheet must be in your Google Drive to run this script.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install saxonche
from saxonche import *

from lxml import etree

from datetime import datetime

!pip install bookops-worldcat
from bookops_worldcat import MetadataSession, WorldcatAccessToken

from io import BytesIO, StringIO

!pip install sickle
from sickle import Sickle
from sickle.iterator import OAIResponseIterator

import requests
import yaml
import csv
import json
import pandas as pd

!pip install xlsxwriter
from xlsxwriter import Workbook

from logging import exception

from google.colab import runtime

# Retrieve OAI metadata

## !! ACTION REQUIRED: input dates and subfolder name when prompted



In [ ]:
date1 = input("Start date (yyyy-mm-dd): ")
date2 = input("End date (yyy-mm-dd): ")

new_folder = input("Subfolder for this semester e.g. '2025_Summer': ")

In [ ]:
# API config file
configfile = "/content/drive/My Drive/ETD2MARC/metadata_api_config.yml"

# location of XSLT file
XSLTfile = "/content/drive/My Drive/ETD2MARC/RRETD2MARC_2026_06_19.xsl"

# OAI metadata that is run through the XSLT script (does not include records with no DOI or with a future embargo date)
saveFile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/all_ETDs.xml"
inputXMLfile = saveFile

# embargoed XML to run at a later date, when embargoes have passed
embargoXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/embargoed_ETDs_"+new_folder+".xml"

# direct output of XSLT script - all MARC/XML records that were run through (not no DOI or embargoed)
outputXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/all_MARC_"+new_folder+".xml"

# only valid MARC/XML records
validXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/valid_MARC_"+new_folder+".xml"
# only invalid MARC/XML records
invalidXMLfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/invalid_MARC_"+new_folder+".xml"

# list of records missing DOIs
missingfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/missingDOIs_list_"+new_folder+".xlsx"
# list of invalid MARC records
invalidfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/invalid_list_"+new_folder+".xlsx"
# list of records with active embargo dates that were not processed
embargofile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/embargo_list_"+new_folder+".xlsx"
# list of campus access only records
campusfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/campusOnly_list_"+new_folder+".xlsx"
# list of records that failed to upload to WMS
errorfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/error_list_"+new_folder+".xlsx"
# list of records successfully uploaded to WMS
processedfile = "/content/drive/My Drive/ETD2MARC/"+new_folder+"/processed_list_"+new_folder+".xlsx"


Save XML as etree

In [ ]:

# OAI call to the RR
sickle = Sickle('https://researchrepository.wvu.edu/do/oai', iterator=OAIResponseIterator)

responses = sickle.ListRecords(**{'metadataPrefix': 'document-export', 'set':'publication:etd',
                                  'from':date1, 'until':date2})

NSMAP = {None : 'http://www.openarchives.org/OAI/2.0/',
        'doc' : ''}

# set up lxml etrees for ones to run through and ones that are embargoed
root = etree.Element("{http://www.openarchives.org/OAI/2.0/}OAI-PMH", nsmap=NSMAP)
records = etree.Element("{http://www.openarchives.org/OAI/2.0/}ListRecords", nsmap=NSMAP)

e_root = etree.Element("{http://www.openarchives.org/OAI/2.0/}OAI-PMH", nsmap=NSMAP)
e_records = etree.Element("{http://www.openarchives.org/OAI/2.0/}ListRecords", nsmap=NSMAP)

# set up lists
missinglist = []
embargolist = {
    'doi':[],
    'embargo_date':[]
}
campuslist = {
    'doi':[],
    'campus_date':[]
}

# iterate through records in the OAI and process
for response in responses:

    xml = response.xml

    for ListRecords in xml:

      for record in ListRecords:
        # check that DOI exists
        if record.xpath(".//*[local-name() = 'field'][@name='doi']"):
          # check if embargo date exists
          if record.xpath(".//*[local-name() = 'field'][@name='embargo_date']"):
              embargo_date = record.xpath(".//*[local-name() = 'field'][@name='embargo_date']/*[local-name() = 'value']/text()")[0][:10]
              # do not process if embargo date is in the future
              if datetime.strptime(embargo_date, '%Y-%m-%d').date() > datetime.today().date():
                embargolist['doi'].append(record.xpath(".//*[local-name() = 'field'][@name='doi']/*[local-name() = 'value']/text()")[0])
                embargolist['embargo_date'].append(embargo_date)
                e_records.append(record)
              else:
                # add campus access only to list
                if "campus" in record.xpath(".//*[local-name() = 'document-type']/text()")[0]:
                  campuslist['doi'].append(record.xpath(".//*[local-name() = 'field'][@name='doi']/*[local-name() = 'value']/text()")[0])
                  campuslist['campus_date'].append(record.xpath(".//*[local-name() = 'submission-date']/text()")[0][:4])
                # otherwise add record to lxml etree to be run through the XSLT
                records.append(record)
          else:
            # otherwise add record to lxml etree to be run through the XSLT
            # add campus access only to list
              if "campus" in record.xpath(".//*[local-name() = 'document-type']/text()")[0]:
                  campuslist['doi'].append(record.xpath(".//*[local-name() = 'field'][@name='doi']/*[local-name() = 'value']/text()")[0])
                  campuslist['campus_date'].append(record.xpath(".//*[local-name() = 'submission-date']/text()")[0][:4])
              records.append(record)
        # do not process records with no DOI - add to list
        elif record.xpath(".//*[local-name() = 'coverpage-url']"):
          missinglist.append(record.xpath(".//*[local-name() = 'coverpage-url']/text()")[0])

# write list of missing DOIs and list of embargos to excel
df = pd.DataFrame(missinglist)
df.to_excel(missingfile, index=False)

df2 = pd.DataFrame(embargolist)
df2.to_excel(embargofile, index=False)

df3 = pd.DataFrame(campuslist)
df3.to_excel(campusfile, index=False)

# add records with DOIs and no or past embargo dates to be processed
root.append(records)
etree.cleanup_namespaces(root)

# save records to be processed as XML file
with open(saveFile, 'wb') as fp:
  fp.write(etree.tostring(root, pretty_print="true", encoding="utf-8"))

# save embargoed records for future processing
e_root.append(e_records)
etree.cleanup_namespaces(e_root)

with open(embargoXMLfile, 'wb') as fp:
  fp.write(etree.tostring(e_root, pretty_print="true", encoding="utf-8"))

## Run XSLT script to produce MARC

In [ ]:
# run XSLT script
proc = PySaxonProcessor(license=False)

xsltproc = proc.new_xslt30_processor()
document = proc.parse_xml(xml_file_name=inputXMLfile)
executable = xsltproc.compile_stylesheet(stylesheet_file=XSLTfile)

output = executable.transform_to_string(xdm_node=document)

# save MARC/XML output to file
with open(outputXMLfile, "w") as file:
    file.write(output)

## API call

In [ ]:
# set up lxml etrees for valid and invalid MARC
marcNSMAP = {'marc' : 'http://www.loc.gov/MARC21/slim'}
invalidRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
validRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)
errorRoot = etree.Element("{http://www.loc.gov/MARC21/slim}collection", nsmap=marcNSMAP)

In [ ]:
# start API session
with open(configfile, 'r') as stream:
    config = yaml.safe_load(stream)

    token = WorldcatAccessToken(
        key= config.get('key'),
        secret= config.get('secret'),
        scopes="WorldCatMetadataAPI:manage_bibs",
    )
    print(token)
    print(token.is_expired())

## VALIDATE MARC RECORD

In [ ]:
# function to retrieve JSON error output from BookOps-WorldCat
# solution from https://www.reddit.com/r/learnpython/comments/s20v6w/easy_way_to_extract_json_part_of_a_string/

def parse_json(s):
    s = s[next(idx for idx, c in enumerate(s) if c in "{["):]
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        return json.loads(s[:e.pos])

In [ ]:
invalidList = {
    'url':[],
    'error':[]
}

# validate record
with open(outputXMLfile,"rb") as xml_file:
    session = MetadataSession(authorization=token)
    marcCollection = BytesIO(xml_file.read())
    tree = etree.parse(marcCollection)
    root = tree.getroot()
    for marcRecord in root.iterfind("{http://www.loc.gov/MARC21/slim}record"):
      # if record validates, add to the valid etree
      try:
        response = session.bib_validate(
        record = etree.tostring(marcRecord),
        recordFormat="application/marcxml+xml",
        validationLevel="validateFull",
        )
        print(response.json())
        if response.json()["status"]["summary"] == 'VALID':
          validRoot.append(marcRecord)
      # otherwise add to the invalid etree and add the id and error to the list of invalid records
      except Exception as e:
        error_json = parse_json(str(e))
        print(error_json)

        invalidRoot.append(marcRecord)
        invalidList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()"))
        invalidList['error'].append(error_json['validationErrors']['errors'])

# save invalid list to file
df = pd.DataFrame(invalidList)
df.to_excel(invalidfile, index=False)

# save MARC/XML to files
with open(invalidXMLfile, 'wb') as fp:
  fp.write(etree.tostring(invalidRoot, pretty_print="true", encoding="utf-8"))

with open(validXMLfile, 'wb') as fp:
  fp.write(etree.tostring(validRoot, pretty_print="true", encoding="utf-8"))


## !!ACTION REQUIRED - enter y to upload MARC records to OCLC

In [ ]:
check = input("Ready to upload via API? (y/n): ")

if check != 'y':
  runtime.unassign()
else:
  errorList = {
      'url':[],
      'error':[]
  }

  processedList = {
      'OCN':[],
      'DOI':[]
  }

  session = MetadataSession(authorization=token)

  for marcRecord in validRoot.iterfind("{http://www.loc.gov/MARC21/slim}record"):
    try:
      createResponse = session.bib_create(
          record = etree.tostring(marcRecord),
          recordFormat="application/marcxml+xml"
              )
      response = createResponse.content
      responseTree = etree.parse(BytesIO(response))
      responseRoot = responseTree.getroot()
      print(responseRoot.xpath("./*[local-name() ='controlfield'][@tag='001']/text()"))
      processedList['OCN'].append(responseRoot.xpath("./*[local-name() ='controlfield'][@tag='001']/text()")[0])
      processedList['DOI'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()")[0])

    except Exception as e:
      error_json = parse_json(str(e))
      print(error_json)

      errorRoot.append(marcRecord)
      errorList['url'].append(marcRecord.xpath("./*[local-name() ='datafield'][@tag='856']/*[local-name() = 'subfield'][@code='u']/text()")[0])
      errorList['error'].append(error_json)

  df = pd.DataFrame(errorList)
  df.to_excel(errorfile, index=False)

  processed_df = pd.DataFrame(processedList)
  processed_df.to_excel(processedfile, index=False)


